# 📦 01. 데이터 수집 (Steam API)

## 목적
Steam Web API를 통해 Dave the Diver의 리뷰를 수집하고,
SQLite DB와 raw JSON으로 저장한다.

## 이 노트북의 결과물
- `data/raw/reviews_all.json` — 전처리용 원본 백업
- `data/dave_diver.db` — SQLite DB (reviews 테이블)
- `data/sample/sample_reviews.csv` — GitHub용 샘플 100건

---
## 1. 라이브러리 & 설정 로드

In [3]:
import sys
import os
import json
import sqlite3
import steamreviews
import pandas as pd
from datetime import datetime

sys.path.append('..')
from config import APP_ID, RAW_DATA_DIR, DB_PATH, SAMPLE_DATA_DIR

os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(SAMPLE_DATA_DIR, exist_ok=True)

print(f"✅ 설정 로드 완료")
print(f"   수집 대상 게임 ID : {APP_ID}")
print(f"   raw 저장 경로     : {RAW_DATA_DIR}")
print(f"   DB 저장 경로      : {DB_PATH}")

✅ 설정 로드 완료
   수집 대상 게임 ID : 1868140
   raw 저장 경로     : data/raw/
   DB 저장 경로      : data/dave_diver.db


---
## 2. Steam API 연결 테스트

In [5]:
# 1건만 받아서 API 연결 상태 + 전체 통계 확인
import requests

url = f"https://store.steampowered.com/appreviews/{APP_ID}"
params = {
    'json'         : 1,
    'filter'       : 'all',   
    'language'     : 'all',
    'review_type'  : 'all',
    'purchase_type': 'all',
    'num_per_page' : 1,       # 딱 1건만
}

#브라우저 메타정보
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

def fmt_num(val):
    return f"{val:,}" if isinstance(val, int) else str(val)

resp = requests.get(url, params=params, headers=headers, timeout=10)
data = resp.json()
qs   = data.get('query_summary', {})
reviews = data.get('reviews', [])

print("\n✅ API 연결 성공!")
print(f"   전체 리뷰 수  : {fmt_num(qs.get('total_reviews',  'N/A'))}")
print(f"   긍정 리뷰     : {fmt_num(qs.get('total_positive', 'N/A'))}")
print(f"   부정 리뷰     : {fmt_num(qs.get('total_negative', 'N/A'))}")
print(f"   평가 요약     : {qs.get('review_score_desc', 'N/A')}")
if reviews:
    print(f"\n   [샘플 리뷰 1건]")
    print(f"   언어      : {reviews[0].get('language')}")
    print(f"   추천 여부 : {reviews[0].get('voted_up')}")
    print(f"   내용 앞부분: {reviews[0].get('review', '')[:60]}...")


✅ API 연결 성공!
   전체 리뷰 수  : 145,875
   긍정 리뷰     : 141,215
   부정 리뷰     : 4,660
   평가 요약     : Overwhelmingly Positive

   [샘플 리뷰 1건]
   언어      : english
   추천 여부 : True
   내용 앞부분: Dave the Diver is one of those rare games that just gets it....


---
## 3. 전체 리뷰 수집

`download_reviews_for_app_id`는 내부적으로:
- cursor 페이지네이션 자동 처리
- Rate Limit 시 자동 쿨다운 후 재시도
- 중복 리뷰 감지 후 자동 종료

> **중단 후 재개 방법**: 수집이 중간에 끊기면 아래 resume-cell의 주석을 풀고 실행한다.

In [7]:
request_params = {
    'language'     : 'all',
    'review_type'  : 'all',
    'purchase_type': 'all',
    'filter'       : 'recent',
}

print("🚀 전체 리뷰 수집 시작...")
print("   cursor / User-Agent / Rate Limit 은 라이브러리가 자동 처리합니다.")
print()

review_dict, query_count = steamreviews.download_reviews_for_app_id(
    APP_ID,
    chosen_request_params=request_params,
    verbose=False, 
)

all_reviews_raw = review_dict['reviews']  # {review_id: review_data}

print(f"\n📦 수집 완료: {len(all_reviews_raw):,}건")
print(f"   API 호출 횟수: {query_count:,}회")

🚀 전체 리뷰 수집 시작...
   cursor / User-Agent / Rate Limit 은 라이브러리가 자동 처리합니다.

[appID = 1868140] expected #reviews = 145875
Number of queries 150 reached. Cooldown: 310 seconds
Number of queries 150 reached. Cooldown: 310 seconds
Number of queries 150 reached. Cooldown: 310 seconds
Number of queries 150 reached. Cooldown: 310 seconds
Number of queries 150 reached. Cooldown: 310 seconds
Number of queries 150 reached. Cooldown: 310 seconds
Number of queries 150 reached. Cooldown: 310 seconds
Number of queries 150 reached. Cooldown: 310 seconds
Number of queries 150 reached. Cooldown: 310 seconds

📦 수집 완료: 145,877건
   API 호출 횟수: 109회


In [ ]:
# ── 중단 후 재개 (정상 완료 시 이 셀 스킵) ──────────────────────────────────
# review_dict = steamreviews.load_review_dict(APP_ID)
# last_cursor = list(review_dict['cursors'].keys())[-1]
# print(f"마지막 커서: {last_cursor}")
#
# review_dict, query_count = steamreviews.download_reviews_for_app_id(
#     APP_ID,
#     chosen_request_params=request_params,
#     start_cursor=last_cursor,
#     verbose=False,
# )
# all_reviews_raw = review_dict['reviews']
# print(f"재개 후 총 {len(all_reviews_raw):,}건")

print("ℹ️  중단 후 재개용 셀입니다. 정상 완료 시 스킵하세요.")

---
## 4. Raw JSON 백업 저장

In [ ]:
# steamreviews는 {review_id: review_data} dict으로 반환 → list로 변환
all_reviews_list = list(all_reviews_raw.values())

raw_path = os.path.join(RAW_DATA_DIR, 'reviews_all.json')
with open(raw_path, 'w', encoding='utf-8') as f:
    json.dump(all_reviews_list, f, ensure_ascii=False, indent=2)

file_size_mb = os.path.getsize(raw_path) / (1024 * 1024)
print(f" Raw JSON 저장 완료")
print(f"   경로  : {raw_path}")
print(f"   건수  : {len(all_reviews_list):,}건")
print(f"   크기  : {file_size_mb:.1f} MB")

✅ Raw JSON 저장 완료
   경로  : data/raw/reviews_all.json
   건수  : 145,877건
   크기  : 171.7 MB


---
## 5. DataFrame 변환 및 전처리

In [9]:
df = pd.json_normalize(all_reviews_list)

print("원본 컬럼 목록:")
for col in df.columns:
    print(f"  {col}")

원본 컬럼 목록:
  recommendationid
  language
  appid
  review
  timestamp_created
  timestamp_updated
  voted_up
  votes_up
  votes_funny
  weighted_vote_score
  comment_count
  steam_purchase
  received_for_free
  refunded
  written_during_early_access
  primarily_steam_deck
  app_release_date
  reactions
  csgo_disclaimer
  author.steamid
  author.personaname
  author.persona_status
  author.profile_url
  author.num_games_owned
  author.num_reviews
  author.playtime_forever
  author.playtime_last_two_weeks
  author.playtime_at_review
  author.last_played
  author.avatar
  author.deck_playtime_at_review
  hardware.manufacturer
  hardware.model
  hardware.dx_video_card
  hardware.dx_vendorid
  hardware.dx_deviceid
  hardware.num_gpu
  hardware.system_ram
  hardware.os
  hardware.cpu_vendor
  hardware.cpu_name
  hardware.gaming_device_type
  hardware.dx_driver_version
  hardware.dx_driver_name
  hardware.adapter_description
  hardware.driver_version
  hardware.driver_date
  hardware.vram_siz

In [ ]:
rename_map = {
    'recommendationid'          : 'review_id',
    'author.steamid'            : 'steam_id',
    'author.playtime_forever'   : 'playtime_forever',
    'author.playtime_at_review' : 'playtime_at_review',
    'author.num_games_owned'    : 'num_games_owned',
    'author.num_reviews'        : 'num_reviews',
    'author.last_played'        : 'last_played',
    'review'                    : 'review_text',
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

df['voted_up']    = df['voted_up'].astype(int)
df['review_date'] = pd.to_datetime(df['timestamp_created'], unit='s').dt.strftime('%Y-%m-%d')
df['review_month']= df['review_date'].str[:7]

df['playtime_hours_forever']   = (df['playtime_forever']   / 60).round(1)
df['playtime_hours_at_review'] = (df['playtime_at_review'] / 60).round(1)

print(f" 변환 완료: {len(df):,}건 / {df.shape[1]}개 컬럼")
print("\n최종 컬럼 목록:")
for col in df.columns:
    print(f"  {col}  ({df[col].dtype})")

✅ 변환 완료: 145,877건 / 54개 컬럼

최종 컬럼 목록:
  review_id  (str)
  language  (str)
  appid  (int64)
  review_text  (str)
  timestamp_created  (int64)
  timestamp_updated  (int64)
  voted_up  (int64)
  votes_up  (int64)
  votes_funny  (int64)
  weighted_vote_score  (object)
  comment_count  (int64)
  steam_purchase  (bool)
  received_for_free  (bool)
  refunded  (bool)
  written_during_early_access  (bool)
  primarily_steam_deck  (bool)
  app_release_date  (str)
  reactions  (object)
  csgo_disclaimer  (bool)
  steam_id  (str)
  author.personaname  (str)
  author.persona_status  (str)
  author.profile_url  (str)
  num_games_owned  (int64)
  num_reviews  (int64)
  playtime_forever  (int64)
  author.playtime_last_two_weeks  (int64)
  playtime_at_review  (int64)
  last_played  (int64)
  author.avatar  (str)
  author.deck_playtime_at_review  (float64)
  hardware.manufacturer  (str)
  hardware.model  (str)
  hardware.dx_video_card  (str)
  hardware.dx_vendorid  (float64)
  hardware.dx_deviceid  (flo

In [11]:
df[['review_id', 'language', 'voted_up',
    'playtime_hours_at_review', 'review_month', 'review_text']].head(5)

,review_id,language,voted_up,playtime_hours_at_review,review_month,review_text
0,218669790,latam,1,24.9,2026-02,Uno de los juegos más hermosos y cautivadores ...
1,218667160,koreana,1,17.5,2026-02,2시간만 하고 환불 할려고 했는데 정신 차리니까 16시간 째 하고 있는 게임
2,218667034,english,1,6.2,2026-02,Sweet! Chum!
3,218665639,koreana,1,25.9,2026-02,잼있어요 굿굿
4,218664822,french,1,4.5,2026-02,jeu chill


---
## 6. SQLite 저장

In [ ]:
db_path = os.path.join('..', DB_PATH)

# list 타입 컬럼 → JSON 문자열로 변환 (SQLite 저장용)
for col in df.columns:
    if df[col].apply(lambda x: isinstance(x, list)).any():
        df[col] = df[col].apply(json.dumps)

conn = sqlite3.connect(db_path)
df.to_sql('reviews', conn, if_exists='replace', index=False)
conn.close()

db_size_mb = os.path.getsize(db_path) / (1024 * 1024)
print(f" SQLite 저장 완료")
print(f"   경로  : {db_path}")
print(f"   건수  : {len(df):,}건")
print(f"   크기  : {db_size_mb:.1f} MB")

✅ SQLite 저장 완료
   경로  : ../data/dave_diver.db
   건수  : 145,877건
   크기  : 74.8 MB


---
## 7. 샘플 데이터 저장 (GitHub용)

In [4]:
# 샘플 재생성 (영어 45건 + 한국어 5건)
conn = sqlite3.connect(os.path.join('..', DB_PATH))

df_en = pd.read_sql("SELECT * FROM reviews WHERE language = 'english' ORDER BY RANDOM() LIMIT 45", conn)
df_ko = pd.read_sql("SELECT * FROM reviews WHERE language = 'korean'  ORDER BY RANDOM() LIMIT 5",  conn)

conn.close()

df_sample = pd.concat([df_en, df_ko]).reset_index(drop=True)

sample_path = os.path.join('..', SAMPLE_DATA_DIR, 'sample_reviews.csv')
df_sample.to_csv(sample_path, index=False, encoding='utf-8-sig')

print(f"영어 {len(df_en)}건 + 한국어 {len(df_ko)}건 → {sample_path}")

영어 45건 + 한국어 0건 → ../data/sample/sample_reviews.csv


---
## 8. 수집 결과 요약

In [5]:
# 수집 결과 요약
conn = sqlite3.connect(os.path.join('..', DB_PATH))

summary = pd.read_sql("""
    SELECT
        COUNT(*)                              AS total,
        SUM(voted_up)                         AS positive,
        SUM(1 - voted_up)                     AS negative,
        ROUND(AVG(voted_up) * 100, 1)         AS positive_rate,
        MIN(review_date)                      AS date_min,
        MAX(review_date)                      AS date_max
    FROM reviews
""", conn).iloc[0]

lang_dist = pd.read_sql("""
    SELECT language, COUNT(*) AS count
    FROM reviews
    GROUP BY language
    ORDER BY count DESC
    LIMIT 10
""", conn)

playtime_med = pd.read_sql("""
    SELECT playtime_hours_at_review FROM reviews
""", conn)['playtime_hours_at_review'].median()

conn.close()

print("=" * 50)
print(f"총 리뷰 수        : {int(summary.total):,}건")
print(f"긍정 리뷰         : {int(summary.positive):,}건 ({summary.positive_rate}%)")
print(f"부정 리뷰         : {int(summary.negative):,}건")
print()
print("언어별 분포 (상위 10개):")
print(lang_dist.to_string(index=False))
print()
print(f"수집 기간         : {summary.date_min} ~ {summary.date_max}")
print(f"평균 플레이타임   : {playtime_med:.1f}시간 (중앙값)")
print("=" * 50)

총 리뷰 수        : 145,877건
긍정 리뷰         : 141,216건 (96.8%)
부정 리뷰         : 4,661건

언어별 분포 (상위 10개):
 language  count
 schinese  54059
  english  51732
  koreana  11318
 tchinese   6690
brazilian   4452
  spanish   3925
   german   3641
   french   1989
  russian   1269
  turkish   1144

수집 기간         : 2022-10-27 ~ 2026-02-19
평균 플레이타임   : 18.6시간 (중앙값)
